# Semana 11: Modelos Complementarios
## Notebook Conceptual (NB1) – Comparación en Datos con Outliers

**Propósito:** Cubrir técnicas adicionales útiles en situaciones específicas, como presencia de outliers o estructuras jerárquicas.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Comparar regresión lineal, HuberRegressor y RANSAC en presencia de outliers.
- Usar SVR con kernel RBF para datos no lineales y comparar con regresión lineal.
- Aplicar clustering jerárquico, generar dendrogramas y comparar cortes con K-Means.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y fijamos la semilla para reproducibilidad.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.linear_model import LinearRegression, HuberRegressor, RANSACRegressor
from sklearn.svm import SVR
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.datasets import make_regression, make_blobs, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, silhouette_score

# Scipy para clustering jerárquico
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Semilla
np.random.seed(42)

print("Librerías importadas correctamente.")

---
## 1. Generación de Datos Sintéticos con Outliers

Creamos datos con una relación lineal y añadimos outliers para comparar la robustez de diferentes modelos de regresión.

In [ ]:
# Parámetros
n_samples = 200
n_outliers = 30

# Datos normales (inliers)
X, y, coef = make_regression(n_samples=n_samples, n_features=1, noise=5, coef=True, random_state=42)

# Añadimos outliers
np.random.seed(42)
X_outliers = np.random.uniform(low=-3, high=3, size=(n_outliers, 1))
y_outliers = np.random.uniform(low=50, high=100, size=n_outliers)

# Combinamos
X_combined = np.vstack([X, X_outliers])
y_combined = np.hstack([y, y_outliers])

# Visualizamos
plt.figure(figsize=(10, 6))
plt.scatter(X, y, c='blue', label='Inliers', alpha=0.6)
plt.scatter(X_outliers, y_outliers, c='red', label='Outliers', alpha=0.6)
plt.xlabel('X')
plt.ylabel('y')
plt.title('Datos sintéticos con outliers')
plt.legend()
plt.grid(True)
plt.show()

---
## 2. Comparación: Regresión Lineal vs HuberRegressor vs RANSAC

Ajustamos los tres modelos y visualizamos las rectas obtenidas.

In [ ]:
# Modelos
lr = LinearRegression()
huber = HuberRegressor(epsilon=1.35)
ransac = RANSACRegressor(min_samples=0.5, residual_threshold=5.0, random_state=42)

# Entrenamos
lr.fit(X_combined, y_combined)
huber.fit(X_combined, y_combined)
ransac.fit(X_combined, y_combined)

# Predicciones para visualización
X_plot = np.linspace(X_combined.min(), X_combined.max(), 100).reshape(-1, 1)
y_lr = lr.predict(X_plot)
y_huber = huber.predict(X_plot)
y_ransac = ransac.predict(X_plot)

# Visualización
plt.figure(figsize=(12, 8))
plt.scatter(X, y, c='blue', label='Inliers', alpha=0.6)
plt.scatter(X_outliers, y_outliers, c='red', label='Outliers', alpha=0.6)
plt.plot(X_plot, y_lr, 'g-', linewidth=2, label=f'Lineal (MSE: {mean_squared_error(y_combined, lr.predict(X_combined)):.1f})')
plt.plot(X_plot, y_huber, 'orange', linewidth=2, label=f'Huber (MSE: {mean_squared_error(y_combined, huber.predict(X_combined)):.1f})')
plt.plot(X_plot, y_ransac, 'purple', linewidth=2, label=f'RANSAC (MSE: {mean_squared_error(y_combined, ransac.predict(X_combined)):.1f})')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Comparación de modelos de regresión en presencia de outliers')
plt.legend()
plt.grid(True)
plt.show()

# Mostramos inliers detectados por RANSAC
inlier_mask = ransac.inlier_mask_
print(f"RANSAC detectó {np.sum(inlier_mask)} inliers de {len(y_combined)} puntos ({np.sum(inlier_mask)/len(y_combined):.1%})")

### Análisis de resultados:
- La **regresión lineal** se ve fuertemente afectada por los outliers, desplazando la recta hacia ellos.
- **HuberRegressor** reduce la influencia de los outliers, proporcionando una estimación más robusta.
- **RANSAC** identifica explícitamente los inliers y ajusta solo con ellos, siendo el más robusto en presencia de muchos outliers.

---
## 3. SVR para Datos No Lineales

Generamos datos con una relación no lineal (función seno con ruido) y comparamos regresión lineal con SVR con kernel RBF.

In [ ]:
# Generamos datos no lineales
np.random.seed(42)
X_nonlin = np.sort(np.random.uniform(-3, 3, 200)).reshape(-1, 1)
y_nonlin = np.sin(X_nonlin).ravel() + np.random.normal(0, 0.1, X_nonlin.shape[0])

# Añadimos algunos outliers
X_nonlin = np.vstack([X_nonlin, np.array([[2.5], [2.7]])])
y_nonlin = np.hstack([y_nonlin, [1.5, 1.8]])

plt.figure(figsize=(10, 6))
plt.scatter(X_nonlin, y_nonlin, alpha=0.6)
plt.xlabel('X')
plt.ylabel('y')
plt.title('Datos no lineales con outliers')
plt.grid(True)
plt.show()

In [ ]:
# Modelos
lr_nonlin = LinearRegression()
svr_lin = SVR(kernel='linear', C=1.0)
svr_rbf = SVR(kernel='rbf', C=1.0, gamma='scale', epsilon=0.1)

# Entrenamos
lr_nonlin.fit(X_nonlin, y_nonlin)
svr_lin.fit(X_nonlin, y_nonlin)
svr_rbf.fit(X_nonlin, y_nonlin)

# Predicciones
X_plot = np.linspace(X_nonlin.min(), X_nonlin.max(), 300).reshape(-1, 1)
y_lr_nonlin = lr_nonlin.predict(X_plot)
y_svr_lin = svr_lin.predict(X_plot)
y_svr_rbf = svr_rbf.predict(X_plot)

# Visualización
plt.figure(figsize=(12, 6))
plt.scatter(X_nonlin, y_nonlin, alpha=0.6, label='Datos')
plt.plot(X_plot, y_lr_nonlin, 'g-', linewidth=2, label='Regresión Lineal')
plt.plot(X_plot, y_svr_lin, 'orange', linewidth=2, label='SVR lineal')
plt.plot(X_plot, y_svr_rbf, 'purple', linewidth=2, label='SVR RBF')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Comparación: Regresión Lineal vs SVR (lineal y RBF)')
plt.legend()
plt.grid(True)
plt.show()

print(f"R² Regresión Lineal: {r2_score(y_nonlin, lr_nonlin.predict(X_nonlin)):.3f}")
print(f"R² SVR lineal: {r2_score(y_nonlin, svr_lin.predict(X_nonlin)):.3f}")
print(f"R² SVR RBF: {r2_score(y_nonlin, svr_rbf.predict(X_nonlin)):.3f}")

---
## 4. Clustering Jerárquico

Generamos datos con estructura de clusters y comparamos clustering jerárquico con K-Means.

In [ ]:
# Generamos datos con 3 clusters
X_clust, y_clust = make_blobs(n_samples=150, centers=3, cluster_std=1.0, random_state=42)

# Estandarizamos
scaler = StandardScaler()
X_clust_scaled = scaler.fit_transform(X_clust)

plt.figure(figsize=(8, 6))
plt.scatter(X_clust_scaled[:, 0], X_clust_scaled[:, 1], c=y_clust, cmap='viridis', alpha=0.6)
plt.title('Datos originales con 3 clusters')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid(True)
plt.show()

### 4.1. Generación de Dendrograma

Usamos `linkage` de scipy para calcular la jerarquía y visualizar el dendrograma.

In [ ]:
# Calculamos la matriz de enlace (linkage) usando método de Ward
Z = linkage(X_clust_scaled, method='ward')

# Visualizamos dendrograma
plt.figure(figsize=(12, 6))
dendrogram(Z, truncate_mode='level', p=5)
plt.title('Dendrograma - Clustering Jerárquico (Ward)')
plt.xlabel('Índice de muestra')
plt.ylabel('Distancia')
plt.axhline(y=10, color='red', linestyle='--', label='Corte en distancia=10')
plt.legend()
plt.show()

In [ ]:
# Cortamos el dendrograma para obtener 3 clusters
clusters_hier = fcluster(Z, t=3, criterion='maxclust')

# Aplicamos K-Means para comparar
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters_kmeans = kmeans.fit_predict(X_clust_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_clust_scaled[:, 0], X_clust_scaled[:, 1], c=clusters_hier, cmap='viridis', alpha=0.6)
axes[0].set_title('Clustering Jerárquico (3 clusters)')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
axes[0].grid(True)

axes[1].scatter(X_clust_scaled[:, 0], X_clust_scaled[:, 1], c=clusters_kmeans, cmap='viridis', alpha=0.6)
axes[1].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], 
                c='red', s=200, marker='X', label='Centroides')
axes[1].set_title('K-Means (3 clusters)')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

### 4.2. Comparación de diferentes tipos de enlace

Probamos distintos métodos de enlace y observamos cómo afectan al dendrograma.

In [ ]:
methods = ['single', 'complete', 'average', 'ward']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for i, method in enumerate(methods):
    Z = linkage(X_clust_scaled, method=method)
    axes[i].set_title(f'Dendrograma - {method.capitalize()} linkage')
    dendrogram(Z, ax=axes[i], truncate_mode='level', p=5)
    axes[i].set_xlabel('Índice de muestra')
    axes[i].set_ylabel('Distancia')

plt.tight_layout()
plt.show()

### 4.3. Evaluación con coeficiente de silueta

Comparamos la calidad de los clusters obtenidos con jerárquico y K-Means.

In [ ]:
sil_hier = silhouette_score(X_clust_scaled, clusters_hier)
sil_kmeans = silhouette_score(X_clust_scaled, clusters_kmeans)

print(f"Coeficiente de silueta - Clustering Jerárquico: {sil_hier:.3f}")
print(f"Coeficiente de silueta - K-Means: {sil_kmeans:.3f}")

---
## 5. Experimentación Adicional: Datos con Formas No Esféricas

Probamos clustering jerárquico en datos con formas no esféricas (lunas) para ver sus limitaciones.

In [ ]:
# Generamos datos de lunas
X_moons, y_moons = make_moons(n_samples=200, noise=0.05, random_state=42)

# Clustering jerárquico
Z_moons = linkage(X_moons, method='ward')
clusters_moons_hier = fcluster(Z_moons, t=2, criterion='maxclust')

# K-Means
kmeans_moons = KMeans(n_clusters=2, random_state=42, n_init=10)
clusters_moons_kmeans = kmeans_moons.fit_predict(X_moons)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='viridis', alpha=0.6)
axes[0].set_title('Datos originales (lunas)')

axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=clusters_moons_hier, cmap='viridis', alpha=0.6)
axes[1].set_title('Clustering Jerárquico (ward)')

axes[2].scatter(X_moons[:, 0], X_moons[:, 1], c=clusters_moons_kmeans, cmap='viridis', alpha=0.6)
axes[2].scatter(kmeans_moons.cluster_centers_[:, 0], kmeans_moons.cluster_centers_[:, 1], 
                c='red', s=200, marker='X', label='Centroides')
axes[2].set_title('K-Means')
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"Silueta Jerárquico (lunas): {silhouette_score(X_moons, clusters_moons_hier):.3f}")
print(f"Silueta K-Means (lunas): {silhouette_score(X_moons, clusters_moons_kmeans):.3f}")

---
## 6. Conclusiones

Hemos explorado modelos complementarios útiles en situaciones específicas:

✔️ **Regresión Robusta**:
- La regresión lineal falla con outliers.
- HuberRegressor reduce su influencia.
- RANSAC identifica inliers y es extremadamente robusto.

✔️ **SVR (Support Vector Regression)**:
- El kernel lineal no captura relaciones no lineales.
- SVR con kernel RBF modela excelentemente datos no lineales.
- Permite controlar la tolerancia al error mediante ε.

✔️ **Clustering Jerárquico**:
- Genera dendrogramas que muestran la jerarquía de clusters.
- Diferentes métodos de enlace (single, complete, average, ward) producen distintas estructuras.
- Permite elegir el número de clusters cortando el dendrograma.
- En datos con formas complejas (lunas), tanto jerárquico como K-Means tienen dificultades.

**Lección clave**: Estas técnicas complementarias amplían nuestro arsenal para abordar problemas específicos: outliers severos, relaciones no lineales y necesidad de jerarquías interpretables.

En el próximo notebook (NB2) aplicaremos estos conceptos a un problema real con datos atípicos.

---
**Fin del Notebook Conceptual - Semana 11**